### Creating Tools and Building Agents

##### The following example demonstrates how to build tools and use them within Chains and Agents

### Example 1 : Building a simple calculator Agent

Let's design a simple application that can interpret natural language math queries and solve them by using modular tools. Specifically, the application should support operations such as adding two numbers and computing the square of a number.
    
    The system should be implemented using both static Chains for predefined logic and dynamic Agents for flexible, query-driven tool selection.

In [3]:
from langchain.agents import create_agent

In [4]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content

'Hello! How can I help you today?'

In [19]:
from langchain.agents import create_agent

In [1]:
## Create Tools with @tool decorator

In [20]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b

@tool
def square_number(x: int) -> int:
    """Return the square of a number."""
    return x*x


In [2]:
## Examine tool's attributes

In [21]:
print(add_numbers.name)
print(add_numbers.description)
print(add_numbers.args)

add_numbers
Add two numbers and return the result.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [22]:
add_numbers.invoke({"a":111117, "b": 5})

111122

#### Tool Usage with Chains:

In LangChain, a chain is used when the number of tool invocations is predetermined. For example, a simple chain can be created to multiply two numbers entered by the user. This involves using a tool for multiplication in a fixed sequence of steps. In this case, the chain would take the user’s inputs, multiply them using the specified tool, and return the result.

#### Tool or function calling allows language models to use external tools by invoking APIs, often called function calls.

In [3]:
## Create Model's instance

In [33]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='amazon.nova-lite-v1:0',
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content

"Hello! How can I assist you today? If you have any questions or need help with a topic, feel free to let me know. Whether it's about science, technology, history, or something else, I'm here to help."

In [34]:
# Bind both tools
llm_with_tools = llm.bind_tools([add_numbers,square_number])

In [ ]:
## Implement a static app with Chains

In [35]:
from operator import itemgetter

# Chain: Ask to add -> extract args -> call add_numbers -> wrap result -> ask to square -> extract args -> call square_number

chain = (
    llm_with_tools
    | (lambda x: x.tool_calls[0]["args"])  # Extract args for add
    | add_numbers                          # Perform addition
    | (lambda result: {"x": result})       # Format result for squaring
    | square_number                        # Perform squaring
)

In [36]:
result = chain.invoke("What is the square of the sum of 44444 and 5?")
print("Final result:", result)


Final result: 1975713601

Using pipe-based tool chains in LangChain offers simplicity for fixed tasks but comes with significant limitations compared to agents. 

    These static chains follow a predefined sequence and lack the ability to reason dynamically, making them unsuitable for handling varied or complex user queries.
        
    They do not support decision-making, error handling, or contextual memory, and become difficult to scale as more tools are added. 
        
    Agents simplify using multiple tools by dynamically selecting and managing them based on the input. Instead of following a fixed sequence, agents decide which tool to use and how many times, offering flexibility.

In [4]:
## Use create_agent to create a dynamic app - which chooses tool based on the query automatically

In [38]:
tools = [add_numbers, square_number]

In [39]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
)

In [40]:
from rich import print

In [44]:
print(agent.invoke(
       {"messages": [{"role": "user", "content": "Square 55 and add 33 with 22 and Find the square of the result."}]
   }
))

{
    'messages': [
        HumanMessage(
            content='Square 55 and add 33 with 22 and Find the square of the result.',
            additional_kwargs={},
            response_metadata={},
            id='7644afdb-7524-47c4-ba97-95d8b4dc5821'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': '<thinking> To complete this task, I will first need to square the number 55. Then, I 
will add 33 and 22 together. Lastly, I will square the result of the addition. Here are the steps:\n\n1. Square 55 
using the "square_number" tool.\n2. Add 33 and 22 using the "add_numbers" tool.\n3. Square the result of the 
addition using the "square_number" tool.\n\nI will start by squaring 55. </thinking>\n'
                },
                {
                    'type': 'tool_use',
                    'name': 'square_number',
                    'input': {'x': 55},
                    'id': 'tooluse_louPg34mIHEpw0YNGM33bU'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': 'd8c88dad-d4e6-43bc-81ea-fbcc50e87c5a',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Mon, 16 Mar 2026 08:32:40 GMT',
                        'content-type': 'application/json',
                        'content-length': '705',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': 'd8c88dad-d4e6-43bc-81ea-fbcc50e87c5a'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [956]},
                'model_provider': 'bedrock_converse',
                'model_name': 'amazon.nova-lite-v1:0'
            },
            id='lc_run--019cf5c6-9977-7510-9503-0a94ca053f3b-0',
            tool_calls=[
                {
                    'name': 'square_number',
                    'args': {'x': 55},
                    'id': 'tooluse_louPg34mIHEpw0YNGM33bU',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 468,
                'output_tokens': 134,
                'total_tokens': 602,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        ToolMessage(
            content='3025',
            name='square_number',
            id='e17f68d8-4783-40a3-8378-a8c4439dab1f',
            tool_call_id='tooluse_louPg34mIHEpw0YNGM33bU'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': '<thinking> I have successfully squared 55, and the result is 3025. Next, I will add 33
and 22 using the "add_numbers" tool. </thinking> '
                },
                {
                    'type': 'tool_use',
                    'name': 'add_numbers',
                    'input': {'a': 33, 'b': 22},
                    'id': 'tooluse_Bf4HLaQ8pCHglwo9NVIj5U'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '5fc378e9-8219-4da3-bf8e-fc5038943ea5',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Mon, 16 Mar 2026 08:32:41 GMT',
                        'content-type': 'application/json',
                        'content-length': '446',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '5fc378e9-8219-4da3-bf8e-fc5038943ea5'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [597]},
                'model_prov

In [47]:
print(agent.invoke(
       {"messages": [{"role": "user", "content": "Explain Agents in LLMs"}]
   }
))

{
    'messages': [
        HumanMessage(
            content='Explain Agents in LLMs',
            additional_kwargs={},
            response_metadata={},
            id='45a20fce-b073-49fb-a4c3-e1ecc3ad10af'
        ),
        AIMessage(
            content='<thinking>The current set of tools does not provide a way to explain technical concepts like 
Agents in LLMs. Therefore, I should respond directly with a brief explanation based on general 
knowledge.</thinking>\nThe concept of Agents in LLMs (Large Language Models) refers to systems that are designed to
interact with users, perform tasks, and make decisions based on the information processed by the LLM. These agents 
can be integrated into applications to automate processes, provide intelligent responses, or assist in 
decision-making. They leverage the capabilities of LLMs to understand and generate human-like text, making them 
useful in various applications such as customer service, content creation, and data analysis.',
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': 'e3bb25c1-3803-4ae2-99c1-80a1363ed5b4',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Mon, 16 Mar 2026 08:35:42 GMT',
                        'content-type': 'application/json',
                        'content-length': '933',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': 'e3bb25c1-3803-4ae2-99c1-80a1363ed5b4'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'end_turn',
                'metrics': {'latencyMs': [1029]},
                'model_provider': 'bedrock_converse',
                'model_name': 'amazon.nova-lite-v1:0'
            },
            id='lc_run--019cf5c9-5ef8-77c3-a7b3-bacbbca4dd71-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 453,
                'output_tokens': 139,
                'total_tokens': 592,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        )
    ]
}

In [48]:
# agent_executor.invoke(
#    {
#        "input": "Add 55 with 22 and Find the square of the result."
#    }
# )

print(agent.invoke(
       {"messages": [{"role": "user", "content": "Add 55 with 22 and Find the square of the result."}]
   }
))

{
    'messages': [
        HumanMessage(
            content='Add 55 with 22 and Find the square of the result.',
            additional_kwargs={},
            response_metadata={},
            id='c2e12823-0c11-456d-9e06-60215847b182'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': '<thinking>To complete this task, I need to first add the numbers 55 and 22. Then, I 
need to find the square of the result from the addition. I can use the "add_numbers" tool for the addition and the 
"square_number" tool for squaring the result.</thinking>\n'
                },
                {
                    'type': 'tool_use',
                    'name': 'add_numbers',
                    'input': {'a': 55, 'b': 22},
                    'id': 'tooluse_aBZ3dMrxNFroeyYvufXCiy'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '84a9715d-8f8d-4eb5-bf30-afbeb97c38fa',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Mon, 16 Mar 2026 08:36:36 GMT',
                        'content-type': 'application/json',
                        'content-length': '569',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '84a9715d-8f8d-4eb5-bf30-afbeb97c38fa'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [718]},
                'model_provider': 'bedrock_converse',
                'model_name': 'amazon.nova-lite-v1:0'
            },
            id='lc_run--019cf5ca-335f-7ef1-b231-86cff96112d2-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 55, 'b': 22},
                    'id': 'tooluse_aBZ3dMrxNFroeyYvufXCiy',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 463,
                'output_tokens': 92,
                'total_tokens': 555,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0}
            }
        ),
        ToolMessage(
            content='77',
            name='add_numbers',
            id='2ef74e3d-3988-4e1f-b3cf-0da9654bb8a3',
            tool_call_id='tooluse_aBZ3dMrxNFroeyYvufXCiy'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': '<thinking>I have the result of adding 55 and 22, which is 77. Now I need to square 
this result. I can use the "square_number" tool for this.</thinking> '
                },
                {
                    'type': 'tool_use',
                    'name': 'square_number',
                    'input': {'x': 77},
                    'id': 'tooluse_ZDMTTuO8iKUKudbFTHd8ZB'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'ResponseMetadata': {
                    'RequestId': '9a792e88-1cf3-4267-ac52-c7d190ea479e',
                    'HTTPStatusCode': 200,
                    'HTTPHeaders': {
                        'date': 'Mon, 16 Mar 2026 08:36:36 GMT',
                        'content-type': 'application/json',
                        'content-length': '457',
                        'connection': 'keep-alive',
                        'x-amzn-requestid': '9a792e88-1cf3-4267-ac52-c7d190ea479e'
                    },
                    'RetryAttempts': 0
                },
                'stopReason': 'tool_use',
                'metrics': {'latencyMs': [612]},
                'model_provider': 'bedrock_converse',
                'model_name': 'amazon.nova-lite-v1:0'
            },
            id='lc_run--019cf5ca-36f2-

####  What is agent_scratchpad?
It stores the agent's internal monologue:

    Tool calls
    
    Outputs from tools
    
    Reasoning steps

This data is fed back into the prompt to help the LLM reason more effectively in subsequent steps.



## Using Legacy - Tool Calling Agent

In [54]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import ConfigurableField
from langchain_core.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' times 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the 'y'."""
    return x**y

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

prompt = ChatPromptTemplate.from_messages([
    ("system", "you're a helpful assistant"), 
    ("human", "{input}"), 
    ("placeholder", "{agent_scratchpad}"),
])

tools = [multiply, exponentiate, add]




agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": "what's 3 plus 5 raised to the 2.743. also what's 17.24 - 918.1241", })



> Entering new AgentExecutor chain...

Invoking: `add` with `{'x': 3, 'y': 5}`
responded: [{'type': 'text', 'text': "<thinking> I need to perform two calculations. First, I need to add 3 and 5, then raise the result to the power of 2.743. Second, I need to subtract 918.1241 from 17.24. I can use the 'add' and 'exponentiate' tools for the first calculation, and the 'add' tool with negative values for the second calculation. </thinking>\n", 'index': 0}, {'type': 'tool_use', 'name': 'add', 'id': 'tooluse_1aKBIMUFHDIoIFD7SUhE8P', 'index': 1, 'input': '{"x":3,"y":5}'}, {'type': 'tool_use', 'name': 'exponentiate', 'id': 'tooluse_yPEsK3c0oyrJvYK0LTmpgO', 'index': 2, 'input': '{"x":8,"y":2.743}'}, {'type': 'tool_use', 'name': 'add', 'id': 'tooluse_5ctDNysCZwu9WWNy12Lr6Q', 'index': 3, 'input': '{"x":17.24,"y":-918.1241}'}]

8.0
Invoking: `exponentiate` with `{'x': 8, 'y': 2.743}`
responded: [{'type': 'text', 'text': "<thinking> I need to perform two calculations. First, I need to add 3 and 5,

{'input': "what's 3 plus 5 raised to the 2.743. also what's 17.24 - 918.1241",
 'output': [{'type': 'text',
   'text': '<thinking> I have the results from the tools. The first calculation was to add 3 and 5, which resulted in 8. Then, I raised this result to the power of 2.743, which gave approximately 300.04. For the second calculation, I subtracted 918.1241 from 17.24, which resulted in -900.88. I can now provide these results to the User. </thinking>\n\nThe result of 3 plus 5 raised to the 2.743 is approximately 300.04. The result of 17.24 minus 918.1241 is -900.88.',
   'index': 0}]}